# 第10章　OCT画像分類の実習：データの入手から誤り分析まで

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 10.1　完成形と、はじめの準備

In [ ]:
import torch
print(torch.cuda.is_available())        # True と出れば成功
print(torch.cuda.get_device_name(0))    # 例: Tesla T4

## 公開データは「そのまま使える」わけではない

In [ ]:
!kaggle datasets download -d <配布ページで確認したスラッグ> -p /content
!unzip -q /content/<ダウンロードしたzipのファイル名>.zip -d /content
!ls /content/OCT2017          # train/ と test/ が見えるか確認する

```text
OCT2017/
├── train/
│   ├── CNV/       CNV-9911627-1.jpeg, ...
│   ├── DME/       DME-1072015-1.jpeg, ...
│   ├── DRUSEN/    DRUSEN-1046140-1.jpeg, ...
│   └── NORMAL/    NORMAL-1017237-1.jpeg, ...
└── test/          （同じ構成）
```

In [ ]:
import glob, os
import pandas as pd

def build_table(root):                                   # フォルダを走査して対応表を作る
    rows = []
    for cls in ["NORMAL", "CNV", "DME", "DRUSEN"]:
        for p in glob.glob(f"{root}/{cls}/*.jpeg"):
            name = os.path.basename(p)                   # 例: CNV-9911627-1.jpeg
            pid  = name.split("-")[1]                    # 患者ID（真ん中の数字）
            rows.append({"path": p, "cls": cls, "patient_id": pid,
                         "label": 0 if cls == "NORMAL" else 1})   # 1 = 異常（要精査）
    return pd.DataFrame(rows)

df    = build_table("/content/OCT2017/train")   # 学習と検証に使う
df_te = build_table("/content/OCT2017/test")   # 最終評価だけに使う（モデルや設定を選ぶためには使わない）
print(df["label"].value_counts())                        # 正常と異常が何枚ずつか
print("患者数 =", df["patient_id"].nunique(), " / テストの患者数 =", df_te["patient_id"].nunique())

## 10.3　学習用・検証用に分ける：患者単位で分割する

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, va_idx = next(gss.split(df, groups=df["patient_id"]))   # 患者IDでまとめて分ける
tr, va = df.iloc[tr_idx], df.iloc[va_idx]

# 分けたら、必ず「混ざっていないこと」を機械的に確かめる
assert set(tr["patient_id"]) & set(va["patient_id"]) == set(), "患者が学習と検証にまたがっている"
# テストも同じ方法で患者の重複を確認する（配布時に患者単位で分割済みでも確認を省かない）
assert not (set(df["patient_id"]) & set(df_te["patient_id"])), "同じ患者が学習/検証とテストにまたがっている"
print(f"学習 {len(tr)}枚 / 検証 {len(va)}枚 / テスト {len(df_te)}枚")
print("学習の陽性率", round(tr['label'].mean(), 3), " 検証の陽性率", round(va['label'].mean(), 3))

## 10.4　前処理とデータの読み込み

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 事前学習(ImageNet)の重みを使うので、学習時と同じ正規化(ImageNetのmean/std)に必ずそろえる
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),        # 本実習では左右反転を使用し、上下反転は使わない
    transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),  # 画像の幅・高さの各2%以内の平行移動
    transforms.ToTensor(), norm,
])
val_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

# ここで Python の「クラス」（class 文）が出てきますが、いまは形だけ眺めて構いません。
# __len__ は「全部で何枚か」、__getitem__ は「i番目の1枚を返す」という約束事で、
# この2つを備えたDatasetを渡せば DataLoader がバッチにまとめます（本書のPythonの章とPyTorchの章で説明します）。
class OCTDataset(Dataset):
    def __init__(self, frame, tf):
        self.f, self.tf = frame.reset_index(drop=True), tf
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        row = self.f.loc[i]
        img = Image.open(row["path"]).convert("RGB")   # グレースケール→3チャネルへ
        return self.tf(img), int(row["label"])

train_loader = DataLoader(OCTDataset(tr,    train_tf), batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(OCTDataset(va,    val_tf),   batch_size=32, num_workers=2)  # モデルを選ぶため
test_loader  = DataLoader(OCTDataset(df_te, val_tf),   batch_size=32, num_workers=2)  # 最終性能を測るため

## 10.5　モデルを用意して学習する

In [ ]:

!pip install timm
import timm, numpy as np
from sklearn.metrics import average_precision_score, recall_score

device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU未選択でも動く
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-4)

for split_name, frame in [("学習", tr), ("検証", va)]:
    assert set(frame["label"].unique()) == {0, 1}, f"{split_name}に正常・異常の両クラスが必要"

counts = np.bincount(tr["label"])                        # クラスごとの画像枚数
w = torch.tensor(counts.sum() / counts, dtype=torch.float32).to(device)
lossf = torch.nn.CrossEntropyLoss(weight=w)              # 少数クラスを重視

best_ap = float("-inf")
best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}  # ループ前に安全な初期値
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = lossf(model(x), y)
        loss.backward()
        opt.step()
    # --- 検証 ---
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in val_loader:
            p = model(x.to(device)).softmax(1)[:, 1].cpu().numpy()   # 「異常」の確率
            ps += list(p); ys += list(y.numpy())
    ap   = average_precision_score(ys, ps)   # AP（Average Precision）。無情報な予測の目安となる陽性率と比較する（有限標本では一致するとは限らない）
    sens = recall_score(ys, (np.array(ps) >= 0.5).astype(int))        # 参考：閾値0.5での感度
    if np.isfinite(ap) and ap > best_ap:                  # 非有限値は採用しない。選ぶ基準はAP
        best_ap = ap
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"epoch {epoch}  AP={ap:.3f}  感度(0.5)={sens:.3f}")

assert np.isfinite(best_ap), "有効な検証が一度も終わっていない"
model.load_state_dict(best_state)                        # 最終epochでなく、検証APが最も高かったモデルを採用

## 10.6　テストデータで評価する：感度・特異度・AUC・混同行列

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

y_true, y_prob = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:                                     # ← 検証ではなくテスト
        p = model(x.to(device)).softmax(1)[:, 1].cpu().numpy()   # 異常の確率
        y_prob += list(p); y_true += list(y.numpy())
y_pred = (np.array(y_prob) >= 0.5).astype(int)

# labels=[0,1] を明示する。片方のクラスしか出てこないと ravel() が4個に分解できずエラーになる
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
print("感度 =", round(tp / (tp + fn), 3), " 特異度 =", round(tn / (tn + fp), 3))
print("ROC-AUC =", round(roc_auc_score(y_true, y_prob), 3))
print("AP  =", round(average_precision_score(y_true, y_prob), 3))
print("混同行列  TN,FP,FN,TP =", tn, fp, fn, tp)

## 10.7　新しい1枚で推論する

In [ ]:
model.eval()                                  # 推論モードへ（下の説明を参照）
path = va.iloc[0]["path"]                     # 検証データから1枚選ぶ
img = val_tf(Image.open(path).convert("RGB"))  # 前処理は「検証用」を使う（学習用ではない）
with torch.no_grad():                         # 勾配を作らない
    prob = model(img[None].to(device)).softmax(1)[0, 1].item()
print(f"異常の確率 = {prob:.2f} →", "要精査" if prob >= 0.5 else "正常範囲")

## 10.8　誤り分析：AIがどこで間違えたかを見る

In [ ]:
import matplotlib.pyplot as plt

# y_true / y_pred / y_prob は 10.6 でテストセットから作った配列なので、
# 画像も同じテストセットから取得する。検証セット(va)の添字を使って取得すると、
# ある症例の画像に別の症例の確率を対応づけることになり、配列の長さが合わない場合はエラーになる。
te2 = df_te.reset_index(drop=True)
fn_idx = [i for i in range(len(te2))
          if y_true[i] == 1 and y_pred[i] == 0]        # 見逃した例（偽陰性）
plt.figure(figsize=(12, 3))
for k, i in enumerate(fn_idx[:4]):
    plt.subplot(1, 4, k + 1)
    plt.imshow(Image.open(te2.loc[i, "path"]), cmap="gray")
    plt.axis("off")
    plt.title(f"FN p={y_prob[i]:.2f}")    # 図中は英語（日本語は□に化ける）
plt.show()